In [1]:
import fitz  # PyMuPDF
#read the pdf
doc=fitz.open("bando.pdf")
text = ""
for page in doc:
    text += page.get_text()
import fitz
import re
from typing import Optional


# ── 1. Extraction ────────────────────────────────────────────────────────────

def extract_text_by_page(pdf_path: str) -> list[str]:
    """Extract text per page, stripping repeated headers/footers heuristically."""
    doc = fitz.open(pdf_path)
    pages = [page.get_text() for page in doc]
    return pages


def clean_page_text(text: str) -> str:
    """
    Clean raw PDF text:
    - Rejoin hyphenated line-breaks (e.g. 'documenta-\ntion' → 'documentation')
    - Collapse mid-sentence line breaks (single \n inside a sentence)
    - Normalize whitespace
    """
    # Rejoin words broken across lines with a hyphen
    text = re.sub(r"-\n(\w)", r"\1", text)
    # Single newline mid-sentence → space (keep double newlines as paragraph breaks)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    # Collapse extra whitespace
    text = re.sub(r" {2,}", " ", text)
    return text.strip()


# ── 2. Paragraph splitting ────────────────────────────────────────────────────

def extract_paragraphs(pages: list[str]) -> list[str]:
    """
    Split on double newlines (true paragraph breaks) rather than every \n.
    Strips page numbers and very short lines (likely headers/footers).
    """
    paragraphs = []
    for page_text in pages:
        cleaned = clean_page_text(page_text)
        # Double newline = real paragraph boundary in most PDFs
        blocks = re.split(r"\n{2,}", cleaned)
        for block in blocks:
            block = block.strip()
            # Drop likely headers/footers: short lines or pure numbers
            if len(block) < 40 or re.fullmatch(r"[\d\s\-–|]+", block):
                continue
            paragraphs.append(block)
    return paragraphs


# ── 3. Semantic chunking ──────────────────────────────────────────────────────

SENTENCE_RE = re.compile(r"(?<=[.!?])\s+")


def split_sentences(text: str, min_len: int = 10) -> list[str]:
    return [s.strip() for s in SENTENCE_RE.split(text) if len(s.strip()) >= min_len]


def merge_into_chunks(
    paragraphs: list[str],
    min_chars: int = 600,
    max_chars: int = 1200,
) -> list[str]:
    """
    Merge short paragraphs into chunks that respect sentence boundaries.
    Splits oversized paragraphs rather than letting them pass through unchecked.
    """
    chunks = []
    buffer = ""

    for para in paragraphs:
        # Oversized single paragraph: split by sentence, flush when full
        if len(para) > max_chars:
            sentences = split_sentences(para)
            for sent in sentences:
                if len(buffer) + len(sent) > max_chars and len(buffer) >= min_chars:
                    chunks.append(buffer.strip())
                    buffer = ""
                buffer += sent + " "
            continue

        if len(buffer) + len(para) > max_chars and len(buffer) >= min_chars:
            chunks.append(buffer.strip())
            buffer = ""

        buffer += para + " "

        if len(buffer) >= min_chars:
            chunks.append(buffer.strip())
            buffer = ""

    if buffer.strip():
        chunks.append(buffer.strip())

    return chunks


# ── 4. Overlap ────────────────────────────────────────────────────────────────

def add_overlap(
    chunks: list[str],
    overlap_sentences: int = 1,
    min_sentence_length: int = 10,
) -> list[str]:
    """Prepend the last N sentences of chunk[i-1] to chunk[i]."""

    def get_tail(text: str, n: int) -> Optional[str]:
        sentences = split_sentences(text, min_len=min_sentence_length)
        if not sentences:
            return None
        return " ".join(sentences[-n:])

    if len(chunks) < 2:
        return chunks

    result = [chunks[0]]
    for i in range(1, len(chunks)):
        tail = get_tail(chunks[i - 1], overlap_sentences)
        result.append(f"{tail} {chunks[i]}" if tail else chunks[i])
    return result


# ── 5. Pipeline ───────────────────────────────────────────────────────────────

def chunk_pdf(
    pdf_path: str,
    min_chars: int = 600,
    max_chars: int = 1200,
    overlap_sentences: int = 1,
) -> list[str]:
    pages = extract_text_by_page(pdf_path)
    paragraphs = extract_paragraphs(pages)
    chunks = merge_into_chunks(paragraphs, min_chars=min_chars, max_chars=max_chars)
    chunks = add_overlap(chunks, overlap_sentences=overlap_sentences)

    print(f"Pages            : {len(pages)}")
    print(f"Raw paragraphs   : {len(paragraphs)}")
    print(f"Chunks           : {len(chunks)}")
    print(f"Avg chunk length : {sum(len(c) for c in chunks) / len(chunks):.0f} chars")

    return chunks


chunks = chunk_pdf("bando.pdf", min_chars=600, max_chars=1200, overlap_sentences=1)

Pages            : 23
Raw paragraphs   : 23
Chunks           : 59
Avg chunk length : 1229 chars


In [2]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import ollama

model=SentenceTransformer('intfloat/multilingual-e5-base')
#format the chunks for embedding
formatted_chunks = [f"passage: {paragraph}" for paragraph in chunks]
#embed the chunks
chunks_embeddings = model.encode(formatted_chunks)
#covnert the embeddings to numpy arrays
chunks_embeddings_np = np.array(chunks_embeddings).astype('float32')
# query_embedding_np = np.array(query_embedding).astype('float32')
#normalize the embeddings
faiss.normalize_L2(chunks_embeddings_np)
#create the faiss index
dimension_chunks = chunks_embeddings_np.shape[1]
index_chunks = faiss.IndexFlatIP(dimension_chunks)  # Inner Product (cosine similarity)
#add the embeddings to the index
index_chunks.add(chunks_embeddings_np)



/Users/oussemahassine/Desktop/RAG/rag_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9852.41it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
#use gradio for a simple interface to ask questions to the RAG system
import gradio as gr

#full chatbot function

def get_text(content):
    """Extract plain text from Gradio's content field (str or list of blocks)."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(block["text"] for block in content if block.get("type") == "text")
    return ""

def rag_chatbot(query,history):
    context_query = query
    if history:
        texts = [
            get_text(h["content"])
            for h in history[-4:]
            if h.get("role") == "user"
        ]
        if texts:
            context_query = f"{' '.join(texts)} {query}"

    query_formatted = f"query: {context_query}"
    query_embedding = model.encode([query_formatted])
    query_embedding_np = np.array(query_embedding).astype('float32')
    faiss.normalize_L2(query_embedding_np)
    D_chunks, I_chunks = index_chunks.search(query_embedding_np, 5)
    context_chunks = [formatted_chunks[I_chunks[0][i]] for i in range(5) if D_chunks[0][i] > 0.75]
    messages=[
        {
            "role": "system",
            "content": ("""You are a document assistant. Your job is to answer questions using retrieved passages from a document.

                        ## Step 1 — Classify the question
                        Decide if the question requires the document to answer, or if it is general knowledge.

                        - If it is **general knowledge**: answer it directly. Do not reference the document or passages.
                        - If it **requires the document**: proceed to Step 2.

                        ## Step 2 — Use the passages
                        You will receive retrieved passages in Italian. Follow these rules:
                        - Extract only the information relevant to the question — do not translate the entire passage.
                        - Answer in English only.
                        - Be concise and direct. Do not mention the passages, the translation process, or your reasoning.
                        - If the passages do not contain enough information to answer, respond with:
                        "The document does not contain enough information to answer this question."
                        - If no passages were retrieved, respond with:
                        "No relevant sections were found in the document for this question."

                        ## Output format
                        Provide only the final answer. No preamble, no explanation of your process."""
            )
        },
        
    ]
    for h in history:
        messages.append({
            "role": h["role"],
            "content": get_text(h["content"])
        })
    messages.append(
                {
                    "role": "user",
                    "content": (
                        f"Question: {query}\n\n"
                        f"Passages:\n"
                        f"{chr(10).join([f'{i+1}. {chunk}' for i, chunk in enumerate(context_chunks)])}"
                    )
                }
            )
    response = ollama.chat(model='llama3',messages=messages,

    stream=True
    )
    partial_response = ""
    for chunk in response:
        if 'message' in chunk and 'content' in chunk['message']:
            partial_response += chunk['message']['content']
        yield partial_response
    

In [14]:
#gradio interface
iface = gr.ChatInterface(fn=rag_chatbot, title="RAG Chatbot")
iface.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [ ]:
#split the text into chunks of paragraphs
paragraphs = text.split("\n")
print(f"Number of paragraphs: {len(paragraphs)}")
#remove empty paragraphs
paragraphs = [p for p in paragraphs if p.strip() != ""]
print(f"Number of paragraphs after removing empty ones: {len(paragraphs)}")
#print all the paragraphs



In [ ]:
#if the paragraphs are too short, append them until charachters are more than 400
cleaned_paragraphs = []
temp_paragraph = ""
for p in paragraphs:
    temp_paragraph += p + " "
    if len(temp_paragraph) > 600:
        cleaned_paragraphs.append(temp_paragraph.strip())
        temp_paragraph = ""
if temp_paragraph:
    cleaned_paragraphs.append(temp_paragraph.strip())
#print length of cleaned paragraphs and average length
print(f"Number of cleaned paragraphs: {len(cleaned_paragraphs)}")
#print all the cleaned paragraphs
for i, p in enumerate(cleaned_paragraphs):
    print(f"Cleaned Paragraph {i}: {p}\n")

In [ ]:
#add some overlap of one sentence between the paragraphs to preserve context
for i in range(1, len(cleaned_paragraphs)):
    overlap_sentence = cleaned_paragraphs[i-1].split(".")[-2] + "."
    cleaned_paragraphs[i] = overlap_sentence + " " + cleaned_paragraphs[i]

print(f"Number of cleaned paragraphs after adding overlap: {len(cleaned_paragraphs)}")
print(f"Average length of cleaned paragraphs after adding overlap: {sum(len(p) for p in cleaned_paragraphs) / len(cleaned_paragraphs)}")
print(f"First cleaned paragraph: {cleaned_paragraphs[0]}")
print(f"Second cleaned paragraph: {cleaned_paragraphs[1]}")
print(f"Third cleaned paragraph: {cleaned_paragraphs[2]}")

In [ ]:
#preview some random cleaned paragraphs
import random
for i in range(5):
    print(f"Cleaned Paragraph {i+1}: {cleaned_paragraphs[random.randint(0, len(cleaned_paragraphs)-1)]}")

Okey now, we move to the embedding

In [ ]:
#since we will be doing english queries to italian documents, we will be using a multilingual embedding model, which is the best option for this use case, and it will also save us the time of translating the documents to english.

In [ ]:
from sentence_transformers import SentenceTransformer
model=SentenceTransformer('intfloat/multilingual-e5-base')


In [ ]:
#format the chunks for embedding
formatted_paragraphs = [f"passage: {paragraph}" for paragraph in cleaned_paragraphs]
print (formatted_paragraphs[0])
print (formatted_paragraphs[1])
print (formatted_paragraphs[2])

In [ ]:
#embed the chunks
paragraphs_embeddings = model.encode(formatted_paragraphs)

In [ ]:
#import faiss for vector search and numpy for handling the embeddings
import faiss
import numpy as np

In [ ]:
#text query to embed
query = "For students in the Department of Biomolecular Sciences (DISB), which documents are mandatory in the application?"
query_formatted = f"query: {query}"
query_embedding = model.encode([query_formatted])

In [ ]:
#covnert the embeddings to numpy arrays
paragraphs_embeddings_np = np.array(paragraphs_embeddings).astype('float32')
query_embedding_np = np.array(query_embedding).astype('float32')

In [ ]:
#normalize the embeddings
faiss.normalize_L2(paragraphs_embeddings_np)
faiss.normalize_L2(query_embedding_np)

In [ ]:
#create the faiss index
dimension_paragraphs = paragraphs_embeddings_np.shape[1]
print(f"Embedding dimension: {dimension_paragraphs}")
index_paragraphs = faiss.IndexFlatIP(dimension_paragraphs)  # Inner Product (cosine similarity)

In [ ]:
#add the embeddings to the index
index_paragraphs.add(paragraphs_embeddings_np)

In [ ]:
#search for top 5 most similar chunks in both indexes
k = 5
D_paragraphs, I_paragraphs = index_paragraphs.search(query_embedding_np, k)

In [ ]:
#print the results
print("Top 5 paragraphs:")
for i in range(k):
    print(f"Chunk {I_paragraphs[0][i]} with similarity {D_paragraphs[0][i]:.4f}")
    print(f"Content: {formatted_paragraphs[I_paragraphs[0][i]]}\n") 

In [ ]:
import ollama
#only use the chunks that have similarity above 0.75 to answer the question.
context_chunks = [formatted_paragraphs[I_paragraphs[0][i]] for i in range(k) if D_paragraphs[0][i] > 0.75]
print(f"Context chunks: {context_chunks}")

In [ ]:

response = ollama.chat(model='llama3', messages=[
    {
        "role": "system",
        "content": (
            "You are a helpful assistant. You will be given passages and a question. "
            "Your job is to extract the answer directly from the passages. "
            "Do NOT say 'I don't know' if the information is present in any passage. "
            "Copy numbers and dates exactly as they appear — never approximate. "
            "Answer only in English."
            "If no passages are passed, say that the document does not contain the information needed to answer the question."
        )
    },
    {
        "role": "user",
        "content": (
            f"Question: {query}\n\n"
            f"Passages:\n"
            f"{chr(10).join([f'{i+1}. {chunk}' for i, chunk in enumerate(context_chunks)])}"

        )
    }
])
print("Retrived passages used for answering the question:")
for i, chunk in enumerate(context_chunks):
    print(f"{i+1}. {chunk}\n")
print("Answer from LLM:")
print(response['message']['content'])  

In [ ]:
""" import ollama

# ── Define prompt strategies ──────────────────────────────────────────
prompt_strategies = {
    "baseline": (
        "You are a helpful assistant for answering questions about the document. "
        "Use the following retrieved passages to answer the question. "
        "Answer only in English. If you don't know the answer, say you don't know."
    ),
    "translate": (
        "You are a helpful assistant for answering questions about the document. "
        "The passages are in Italian. First translate each passage to English, "
        "then use the translated text to answer the question. "
        "Answer only in English. If you don't know the answer, say you don't know."
    ),
    "chain_of_thought": (
        "You are a helpful assistant for answering questions about the document. "
        "Use the following retrieved passages to answer the question. "
        "Step 1: Read all passages carefully. "
        "Step 2: Identify which passage contains the most relevant information. "
        "Step 3: Use that information to answer the question. "
        "Answer only in English. If the answer is explicitly present in any passage, you MUST use it."
    ),
    "strict_extraction": (
        "You are a helpful assistant. You will be given passages and a question. "
        "Your job is to extract the answer directly from the passages. "
        "Do NOT say 'I don't know' if the information is present in any passage. "
        "Copy numbers and dates exactly as they appear — never approximate. "
        "Answer only in English."
    ),
    "translate_then_extract": (
        "You are a helpful assistant. You will be given Italian passages and a question in English. "
        "Step 1: Mentally translate all passages to English. "
        "Step 2: Scan every passage for the answer — do not skip any. "
        "Step 3: If the answer is found, state it clearly and concisely. "
        "Step 4: Only say 'I don't know' if none of the passages contain relevant information. "
        "Never approximate numbers — copy them exactly."
    ),
    "role_play": (
        "You are an expert document analyst fluent in both Italian and English. "
        "You are given excerpts from an Italian university document. "
        "Read all passages thoroughly, then answer the question in English. "
        "If a passage contains the answer even partially, use it. "
        "Reproduce numbers, dates, and names exactly as written."
    ),
}

# ── Test questions (focus on the ones that failed before) ─────────────
test_queries = {
    "Q02 - Total scholarships":     "How many scholarships are available for the academic year 2026/2027?",
    "Q03 - Grant Spain":            "What is the monthly grant amount for students going to Spain?",
    "Q07 - Minori opportunità":     "What is the additional monthly grant for students with fewer opportunities during long-duration mobility?",
    "Q09 - Score grade 108":        "What score does a student with a graduation grade of 108 receive in the merit-based scoring?",
    "Q19 - DISB mandatory docs":    "For students in the Department of Biomolecular Sciences (DISB), which documents are mandatory in the application?",
}

# ── Helper ────────────────────────────────────────────────────────────
def ask(system_prompt: str, question: str) -> str:
    q_emb = np.array(model.encode([f"query: {question}"])).astype('float32')
    faiss.normalize_L2(q_emb)
    _, I = index_paragraphs.search(q_emb, 4)

    passages = "\n".join(
        f"{i+1}. {formatted_paragraphs[I[0][i]]}" for i in range(4)
    )
    response = ollama.chat(model='llama3', messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": f"Question: {question}\n\nPassages:\n{passages}"}
    ])
    return response['message']['content'].strip()

# ── Run grid ──────────────────────────────────────────────────────────
results = {q_label: {} for q_label in test_queries}

for q_label, question in test_queries.items():
    print(f"\n{'='*70}")
    print(f"  {q_label}")
    print(f"  Q: {question}")
    print(f"{'='*70}")
    for strategy_name, system_prompt in prompt_strategies.items():
        answer = ask(system_prompt, question)
        results[q_label][strategy_name] = answer
        print(f"\n  [{strategy_name}]")
        print(f"  ➤  {answer}")

# ── Summary table ─────────────────────────────────────────────────────
print("\n\n" + "="*70)
print("SUMMARY — answers per strategy per question")
print("="*70)
header = f"{'Question':<28}" + "".join(f"{s:<22}" for s in prompt_strategies)
print(header)
print("-"*70)
for q_label, strategy_answers in results.items():
    row = f"{q_label:<28}"
    for strategy_name in prompt_strategies:
        # Truncate answer to 18 chars for the table
        short = strategy_answers[strategy_name][:18].replace("\n", " ")
        row += f"{short:<22}"
    print(row) """

In [ ]:
""" # ─────────────────────────────────────────────
# RAG EVALUATION – 20 Questions & Expected Answers
# ─────────────────────────────────────────────
import re

test_questions = [
    # Factual / Direct Lookup
    "What is the application deadline for the Erasmus+ Studio 2026/2027 scholarship?",
    "How many scholarships are available for the academic year 2026/2027?",
    "What is the monthly grant amount for students going to Spain?",
    "What is the minimum duration of physical mobility for a long-duration Erasmus+ period?",
    "What is the maximum number of destinations a student can select in their application?",
    # Numerical / Table Reasoning
    "How many CFUs must a student earn abroad if their stay lasts 5 months?",
    "What is the additional monthly grant for students with fewer opportunities (minori opportunità) during long-duration mobility?",
    "What travel contribution does a student receive for an ecological round trip of 600 km?",
    "What score does a student with a graduation grade of 108 receive in the merit-based scoring?",
    "What is the maximum additional score a department can assign beyond the merit score?",
    # Eligibility / Conditions
    "Can a PhD student apply for a short-duration mobility of 20 days? What condition must be met?",
    "Under what conditions must a student return the full or partial scholarship grant?",
    "Can a student perform Erasmus+ mobility in their own country of residence?",
    "What happens if a student graduates before the end of their mobility period?",
    # Process / Procedural
    "Through which platform must the application be submitted, and what are the steps?",
    "What documents must a student submit to the International Mobility Office at the end of their mobility?",
    "What is a learning agreement (accordo didattico) and when must it be submitted?",
    "What is the OLS test, and when must it be taken?",
    # Department-Specific
    "For students in the Department of Biomolecular Sciences (DISB), which documents are mandatory in the application?",
    "How does the School of Foreign Languages determine its ranking, and does it conduct interviews?",
]

def get_rag_answer(question: str) -> str:
    """Run the full RAG pipeline for a single question."""
    # 1. Embed the question
    q_embedding = model.encode([f"query: {question}"])
    q_embedding_np = np.array(q_embedding).astype('float32')
    faiss.normalize_L2(q_embedding_np)

    # 2. Retrieve top-4 passages
    _, I = index_paragraphs.search(q_embedding_np, 4)

    # 3. Build prompt and call Ollama
    response = ollama.chat(model='llama3', messages=[
        {
            "role": "system",
            "content": (
                "You are a helpful assistant for answering questions about the document. "
                "Use the following retrieved passages to answer the question. "
                "Answer only in English. If you don't know the answer, say you don't know. "
                "Do not mention the passages in your answer, just use them as context."
            )
        },
        {
            "role": "user",
            "content": (
                f"Question: {question}\n\n"
                f"Passages:\n"
                f"1. {formatted_paragraphs[I[0][0]]}\n"
                f"2. {formatted_paragraphs[I[0][1]]}\n"
                f"3. {formatted_paragraphs[I[0][2]]}\n"
                f"4. {formatted_paragraphs[I[0][3]]}"
            )
        }
    ])
    return response['message']['content'].strip()


# ─── Run evaluation ───────────────────────────
print("=" * 70)
print("RAG EVALUATION REPORT")
print("=" * 70)

results = []
for idx, question in enumerate(test_questions, 1):
    print(f"\n[Q{idx:02d}] {question}")
    print("-" * 70)
    try:
        answer = get_rag_answer(question)
        print(f"  ➤  {answer}")
        results.append({"question": question, "answer": answer, "status": "OK"})
    except Exception as e:
        print(f"  ✗  ERROR: {e}")
        results.append({"question": question, "answer": "", "status": f"ERROR: {e}"})

# ─── Summary ──────────────────────────────────
print("\n" + "=" * 70)
ok_count = sum(1 for r in results if r["status"] == "OK")
print(f"SUMMARY: {ok_count}/{len(results)} questions answered successfully.")
print("=" * 70) """

In [ ]:
import fitz  # PyMuPDF
import time 
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import ollama
doc=fitz.open("bando.pdf")
text = ""
for page in doc:
    text += page.get_text()
paragraphs = text.split("\n")
#remove empty paragraphs
paragraphs = [p for p in paragraphs if p.strip() != ""]
#if the paragraphs are too short, append them until charachters are more than 400
cleaned_paragraphs = []
temp_paragraph = ""
for p in paragraphs:
    temp_paragraph += p + " "
    if len(temp_paragraph) > 600:
        cleaned_paragraphs.append(temp_paragraph.strip())
        temp_paragraph = ""
if temp_paragraph:
    cleaned_paragraphs.append(temp_paragraph.strip())
#add some overlap of one sentence between the paragraphs to preserve context
for i in range(1, len(cleaned_paragraphs)):
    overlap_sentence = cleaned_paragraphs[i-1].split(".")[-2] + "."
    cleaned_paragraphs[i] = overlap_sentence + " " + cleaned_paragraphs[i]
model=SentenceTransformer('intfloat/multilingual-e5-base')
#format the chunks for embedding
formatted_paragraphs = [f"passage: {paragraph}" for paragraph in cleaned_paragraphs]
#embed the chunks
paragraphs_embeddings = model.encode(formatted_paragraphs)
#text query to embed
# query_formatted = f"query: {query}"
# query_embedding = model.encode([query_formatted])
#covnert the embeddings to numpy arrays
paragraphs_embeddings_np = np.array(paragraphs_embeddings).astype('float32')
# query_embedding_np = np.array(query_embedding).astype('float32')
#normalize the embeddings
faiss.normalize_L2(paragraphs_embeddings_np)
# faiss.normalize_L2(query_embedding_np)
#create the faiss index
dimension_paragraphs = paragraphs_embeddings_np.shape[1]
index_paragraphs = faiss.IndexFlatIP(dimension_paragraphs)  # Inner Product (cosine similarity)
#add the embeddings to the index
index_paragraphs.add(paragraphs_embeddings_np)
#search for top 5 most similar chunks in both indexes
# D_paragraphs, I_paragraphs = index_paragraphs.search(query_embedding_np, 5)
#only use the chunks that have similarity above 0.75 to answer the question.
# context_chunks = [formatted_paragraphs[I_paragraphs[0][i]] for i in range(5) if D_paragraphs[0][i] > 0.75]

In [ ]:
#gradio interface
iface = gr.Interface(fn=rag_chatbot, inputs="text", outputs="text", title="RAG Chatbot")
iface.launch()